In [1]:
import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd

from PIL import Image

from sklearn.metrics import f1_score

from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
import time

d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
CFG = {
    "model_name": "convnextv2_tiny.fcmae_ft_in22k_in1k",
    "img_size":384,
    "batch_size":32,
    "epochs":50,
    "patience":7,
    "lr":3e-4,
    "weight_decay":1e-4,
    "num_classes":3,
    "num_workers":0,
    "fold":0,
    "device":"cuda"
}

In [4]:
train_transform = A.Compose([

    A.LongestMaxSize(max_size=CFG["img_size"]),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.HorizontalFlip(p=0.5),

    A.Rotate(limit=15, p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=15,
        val_shift_limit=10,
        p=0.3
    ),

    A.CLAHE(
        clip_limit=2,
        tile_grid_size=(8,8),
        p=0.5,
    ),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=10,
        p=0.5
    ),

    A.CoarseDropout(
        num_holes_range=(1,4),
        hole_height_range=(20,50),
        hole_width_range=(20,50),
        p=0.3
    ),

    A.Normalize(),

    ToTensorV2()
])

d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
valid_transform = A.Compose([

    A.LongestMaxSize(max_size=CFG["img_size"]),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.Normalize(),

    ToTensorV2()

])

In [6]:
class WasteDataset(Dataset):

    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row.filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        label = row.label
        return image, label

In [7]:
df = pd.read_csv("train_folds.csv")

print(df.shape)
df.head()

(24853, 4)


,filepath,label,class_name,fold
0,..\data\train\0_Recyclable\R_1.jpg,0,0_Recyclable,2
1,..\data\train\0_Recyclable\R_10.jpg,0,0_Recyclable,2
2,..\data\train\0_Recyclable\R_100.jpg,0,0_Recyclable,2
3,..\data\train\0_Recyclable\R_1000.jpg,0,0_Recyclable,3
4,..\data\train\0_Recyclable\R_1001.jpg,0,0_Recyclable,3


In [8]:
fold = CFG["fold"]

train_df = df[df.fold != fold]
valid_df = df[df.fold == fold]

print(train_df.shape)
print(valid_df.shape)

(19882, 4)
(4971, 4)


In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

print(class_weights)

tensor([0.8752, 2.1079, 0.7230], device='cuda:0')


In [10]:
train_dataset = WasteDataset(
    train_df,
    transforms=train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    transforms=valid_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [11]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
)

model = model.to(CFG["device"])

In [12]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

In [13]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

In [14]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

In [15]:
scaler = torch.amp.GradScaler("cuda")

In [16]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):

    model.train()

    running_loss = 0

    pbar = tqdm(loader, desc="Train")

    for images, labels in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader)

In [17]:
def valid_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0

    preds = []
    labels_all = []

    pbar = tqdm(loader, desc="Valid")

    with torch.no_grad():

        for images, labels in pbar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            pred = outputs.argmax(1)

            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    macro_f1 = f1_score(
        labels_all,
        preds,
        average="macro"
    )

    return running_loss / len(loader), macro_f1

In [18]:
print(len(train_loader))
print(len(valid_loader))

622
156


In [19]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [01:41<00:00,  1.54it/s, loss=0.3440]


Epoch [1/50] | Train Loss: 0.3530 | Valid Loss: 0.3248 | Macro F1: 0.9587 | Time: 851.5s
Best model saved! Macro F1 = 0.95867


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.4114]


Epoch [2/50] | Train Loss: 0.2987 | Valid Loss: 0.3319 | Macro F1: 0.9570 | Time: 539.5s
No improvement (1/7)


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.3204]


Epoch [3/50] | Train Loss: 0.2817 | Valid Loss: 0.3132 | Macro F1: 0.9642 | Time: 536.1s
Best model saved! Macro F1 = 0.96422


... (84 outputs left out)

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.98it/s, loss=0.2947]


Epoch [46/50] | Train Loss: 0.2061 | Valid Loss: 0.2895 | Macro F1: 0.9803 | Time: 531.7s
No improvement (6/7)


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.98it/s, loss=0.2947]


Epoch [47/50] | Train Loss: 0.2063 | Valid Loss: 0.2888 | Macro F1: 0.9812 | Time: 531.4s
No improvement (7/7)
Early Stopping
Training Finished!
Best Macro F1 : 0.98246


In [20]:
CFG["fold"] = 1

In [21]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

print(train_df.shape)
print(valid_df.shape)

(19882, 4)
(4971, 4)


In [22]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [23]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [24]:
print(len(train_loader))
print(len(valid_loader))

622
156


In [25]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

print(class_weights)

tensor([0.8751, 2.1079, 0.7231], device='cuda:0')


In [26]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [27]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

In [29]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

In [30]:
scaler = torch.amp.GradScaler("cuda")

In [31]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.6701]


Epoch [1/50] | Train Loss: 0.3505 | Valid Loss: 0.3175 | Macro F1: 0.9629 | Time: 530.9s
Best model saved! Macro F1 = 0.96286


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.6548]


Epoch [2/50] | Train Loss: 0.3012 | Valid Loss: 0.3249 | Macro F1: 0.9655 | Time: 531.0s
Best model saved! Macro F1 = 0.96548


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.8291]


Epoch [3/50] | Train Loss: 0.2835 | Valid Loss: 0.3306 | Macro F1: 0.9594 | Time: 531.1s
No improvement (1/7)


... (26 outputs left out)

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.6366]


Epoch [17/50] | Train Loss: 0.2248 | Valid Loss: 0.3117 | Macro F1: 0.9736 | Time: 534.0s
No improvement (6/7)


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.6902]


Epoch [18/50] | Train Loss: 0.2299 | Valid Loss: 0.3095 | Macro F1: 0.9664 | Time: 531.3s
No improvement (7/7)
Early Stopping
Training Finished!
Best Macro F1 : 0.97442


In [32]:
CFG["fold"] = 2

In [33]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [34]:
train_dataset = WasteDataset(train_df, train_transform)
valid_dataset = WasteDataset(valid_df, valid_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [35]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [36]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [37]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [38]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.5154]


Epoch [1/50] | Train Loss: 0.3521 | Valid Loss: 0.3368 | Macro F1: 0.9535 | Time: 531.4s
Best model saved! Macro F1 = 0.95347


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.7842]


Epoch [2/50] | Train Loss: 0.3019 | Valid Loss: 0.3235 | Macro F1: 0.9640 | Time: 531.9s
Best model saved! Macro F1 = 0.96397


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.6805]


Epoch [3/50] | Train Loss: 0.2850 | Valid Loss: 0.3174 | Macro F1: 0.9675 | Time: 531.9s
Best model saved! Macro F1 = 0.96753


... (28 outputs left out)

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.3088]


Epoch [18/50] | Train Loss: 0.2257 | Valid Loss: 0.3437 | Macro F1: 0.9569 | Time: 532.8s
No improvement (6/7)


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.4130]


Epoch [19/50] | Train Loss: 0.2219 | Valid Loss: 0.3249 | Macro F1: 0.9656 | Time: 532.4s
No improvement (7/7)
Early Stopping
Training Finished!
Best Macro F1 : 0.97279


In [39]:
CFG["fold"] = 3

In [40]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [41]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [42]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [43]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [44]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [45]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [46]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:54<00:00,  2.89it/s, loss=0.6542]


Epoch [1/50] | Train Loss: 0.3587 | Valid Loss: 0.3541 | Macro F1: 0.9430 | Time: 538.6s
Best model saved! Macro F1 = 0.94296


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.94it/s, loss=0.4707]


Epoch [2/50] | Train Loss: 0.3017 | Valid Loss: 0.3393 | Macro F1: 0.9525 | Time: 533.6s
Best model saved! Macro F1 = 0.95254


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.94it/s, loss=0.7428]


Epoch [3/50] | Train Loss: 0.2865 | Valid Loss: 0.3420 | Macro F1: 0.9574 | Time: 532.7s
Best model saved! Macro F1 = 0.95743


... (30 outputs left out)

Valid: 100%|██████████| 156/156 [00:53<00:00,  2.92it/s, loss=0.7292]


Epoch [19/50] | Train Loss: 0.2259 | Valid Loss: 0.3285 | Macro F1: 0.9688 | Time: 537.5s
No improvement (6/7)


Valid: 100%|██████████| 156/156 [00:53<00:00,  2.92it/s, loss=0.6357]


Epoch [20/50] | Train Loss: 0.2227 | Valid Loss: 0.3282 | Macro F1: 0.9630 | Time: 538.4s
No improvement (7/7)
Early Stopping
Training Finished!
Best Macro F1 : 0.97286


In [47]:
CFG["fold"] = 4

In [48]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [49]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [50]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [51]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [52]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [53]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [54]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [01:21<00:00,  1.92it/s, loss=0.3178]


Epoch [1/50] | Train Loss: 0.3813 | Valid Loss: 0.3568 | Macro F1: 0.9498 | Time: 706.0s
Best model saved! Macro F1 = 0.94976


Valid: 100%|██████████| 156/156 [00:53<00:00,  2.92it/s, loss=0.2961]


Epoch [2/50] | Train Loss: 0.2969 | Valid Loss: 0.3179 | Macro F1: 0.9676 | Time: 537.6s
Best model saved! Macro F1 = 0.96763


Valid: 100%|██████████| 156/156 [00:52<00:00,  2.96it/s, loss=0.3259]


Epoch [3/50] | Train Loss: 0.2897 | Valid Loss: 0.3357 | Macro F1: 0.9640 | Time: 533.0s
No improvement (1/7)


... (8 outputs left out)

Valid: 100%|██████████| 156/156 [00:52<00:00,  2.95it/s, loss=0.3194]


Epoch [8/50] | Train Loss: 0.2528 | Valid Loss: 0.3387 | Macro F1: 0.9543 | Time: 536.0s
No improvement (6/7)


Valid: 100%|██████████| 156/156 [00:53<00:00,  2.94it/s, loss=0.3012]


Epoch [9/50] | Train Loss: 0.2557 | Valid Loss: 0.3361 | Macro F1: 0.9611 | Time: 537.2s
No improvement (7/7)
Early Stopping
Training Finished!
Best Macro F1 : 0.96763
